# MCMC — Exercises

Companion to the note [MCMC](MCMC.md).

Practise Markov chain Monte Carlo: Metropolis–Hastings (symmetric and asymmetric proposals), detailed balance, Gibbs sampling, Hamiltonian Monte Carlo, importance and rejection sampling, and the diagnostics from the note (trace plots, effective sample size, $\hat R$, burn-in). By hand first, then samplers written from scratch in NumPy and checked against exact answers.

**17 exercises** · 🧠 Concept ×4 · ✍️ By hand ×6 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy import stats
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
# Shared target: Beta(9, 5) posterior of a coin (7 heads in 10 tosses, Beta(2,2) prior), known only up to a constant
def log_p_tilde(theta):
    theta = np.asarray(theta, dtype=float)
    out = np.full(theta.shape, -np.inf)
    ok = (theta > 0) & (theta < 1)
    out[ok] = 8 * np.log(theta[ok]) + 4 * np.log(1 - theta[ok])
    return out if out.ndim else float(out)

## Part A · Concepts

### Exercise 1 · Why the normalizer cancels
*🧠 Concept · ★☆☆*

The note says MCMC samples $p(\theta\mid D)$ "without knowing $Z$". Show where $Z=p(D)$ cancels in the Metropolis–Hastings acceptance
ratio. Why is computing $Z$ hard in the first place, and which other quantities can you then estimate from the samples?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$p(\theta\mid D)=\tilde p(\theta)/Z$ with $\tilde p(\theta)=p(D\mid\theta)p(\theta)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\frac{p(\theta'\mid D)}{p(\theta\mid D)}=\frac{\tilde p(\theta')/Z}{\tilde p(\theta)/Z}=\frac{\tilde p(\theta')}{\tilde p(\theta)}$: only *ratios* of the target enter the acceptance probability.
$Z=\int p(D\mid\theta)p(\theta)d\theta$ is a high-dimensional integral with no closed form outside conjugate models ([[Bayesian Inference]]); grids scale exponentially with dimension.
From samples $\theta^{(s)}$ we estimate any posterior expectation $\mathbb E[f(\theta)\mid D]\approx\frac1S\sum_sf(\theta^{(s)})$: means, variances, quantiles/credible intervals, predictive distributions. ($Z$ itself is hard to get from MCMC output.)

</details>

### Exercise 2 · Reading diagnostics
*🧠 Concept · ★☆☆*

Match each symptom with its diagnosis and a fix:
(a) the trace plot starts at 50 and slowly drifts down to around 0 over the first 500 iterations;
(b) the trace looks like a slowly wandering random walk ("snake"), ESS = 40 out of 10 000;
(c) four chains started at different points each look stable, but around different values, $\hat R=1.6$;
(d) the trace is flat for long stretches (same value repeated), acceptance rate 2%.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Burn-in, autocorrelation, multimodality / non-convergence, step size too large.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Burn-in / warm-up**: the chain had not reached the typical set. Discard the first iterations (or start closer).
(b) **High autocorrelation**: steps too small or strongly correlated parameters. Larger steps, reparameterize, use HMC/NUTS; thinning only reduces storage, it does not add information.
(c) **Non-convergence / multimodality**: the chains explore different modes or have not mixed. Run longer, better initialization, tempering, or rethink the model; never pool such chains.
(d) **Step size too large**: almost every proposal lands in low density and is rejected. Reduce the proposal scale (aim for about 20–50% acceptance for random-walk MH).

</details>

### Exercise 3 · The step-size trade-off
*🧠 Concept · ★★☆*

For random-walk Metropolis with proposal $\theta'=\theta+s\,\varepsilon$, $\varepsilon\sim\mathcal N(0,I)$, explain what happens to the acceptance
rate and to the autocorrelation when $s$ is (a) tiny and (b) huge. Why do both extremes give a small effective sample size, and
how do HMC/NUTS (note's table) escape this trade-off in high dimension?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Small steps are almost always accepted but barely move; large steps would move far but are almost always rejected.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Tiny $s$: acceptance $\to1$, but each move is tiny, so consecutive samples are almost identical: huge autocorrelation, small ESS (diffusive random walk; covering a distance $L$ takes $\sim(L/s)^2$ steps).
(b) Huge $s$: proposals land in the tails and are rejected; the chain sits still for long stretches: again high autocorrelation.
The optimum lies in between (for Gaussian targets in high dimension, $s\approx2.38/\sqrt d$ and about 23% acceptance), and the needed number of steps grows with $d$.
HMC uses the gradient $\nabla\log\tilde p$ to simulate Hamiltonian dynamics: long, directed trajectories along the typical set with high acceptance, so it scales much better with dimension; NUTS tunes the trajectory length automatically.

</details>

### Exercise 4 · MCMC vs the alternatives
*🧠 Concept · ★☆☆*

The note's table lists importance/rejection sampling as "simple but poor in high dimensions", and compares MCMC to [[Variational Inference]]
("slower but asymptotically exact"). Explain both statements briefly.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

For rejection sampling think of the acceptance rate when both proposal and target are $d$-dimensional Gaussians with slightly different widths.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Rejection and importance sampling draw independent samples from a fixed proposal $q$. In $d$ dimensions any mismatch between $q$ and $p$ multiplies across dimensions: the acceptance rate (rejection) or the effective number of importance samples decays **exponentially** in $d$ (one weight dominates).
MCMC instead makes *local* moves that adapt to where the chain currently is.
VI turns inference into optimization over a restricted family $q$: fast and scalable, but biased (e.g. it under-estimates variance) no matter how long you run. MCMC's estimates converge to the exact posterior expectations as the number of samples grows (under ergodicity), but it is slower and harder to scale to huge data.

</details>

## Part B · By hand

### Exercise 5 · An MH acceptance probability
*✍️ By hand · ★☆☆*

Target $\tilde p(\theta)=\exp(-\theta^2/2)$, symmetric random-walk proposal. The chain is at $\theta=1$ and proposes $\theta'=2$.
Compute the acceptance probability. What is it for a proposal $\theta'=0.5$?

In [ ]:
acc_to_2 = None
acc_to_05 = None

_a = lambda t, tp: min(1.0, stats.norm.pdf(tp) / stats.norm.pdf(t))
check('accept 1 -> 2', acc_to_2, _a(1, 2)); check('accept 1 -> 0.5', acc_to_05, _a(1, 0.5))

<details>
<summary><b>💡 Hint</b></summary>

With a symmetric $q$ the $q$ terms cancel: $\alpha=\min(1,\tilde p(\theta')/\tilde p(\theta))$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\alpha(1\to2)=\min(1,e^{-(4-1)/2})=e^{-1.5}\approx0.223$. $\alpha(1\to0.5)=\min(1,e^{(1-0.25)/2})=1$: moves uphill are always accepted.

```python
acc_to_2 = np.exp(-1.5)
acc_to_05 = 1.0
```

</details>

### Exercise 6 · Hastings correction for an asymmetric proposal
*✍️ By hand · ★★☆*

Target $\tilde p(\theta)=e^{-\theta}$ on $\theta>0$. Proposal: $\theta'\sim\mathrm{Exponential}$ with mean $\theta$, i.e. $q(\theta'\mid\theta)=\frac1\theta e^{-\theta'/\theta}$.
From $\theta=1$ the proposal is $\theta'=2$. Compute the full MH acceptance probability, and the (wrong) one you get if you forget the $q$ ratio.

In [ ]:
acc_mh = None
acc_wrong = None

_pt = lambda t: np.exp(-t); _q = lambda tp, t: stats.expon(scale=t).pdf(tp)
check('MH acceptance', acc_mh, min(1, _pt(2) * _q(1, 2) / (_pt(1) * _q(2, 1))))
check('without Hastings correction', acc_wrong, min(1, _pt(2) / _pt(1)))

<details>
<summary><b>💡 Hint</b></summary>

$q(1\mid2)=\tfrac12e^{-1/2}$, $q(2\mid1)=e^{-2}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\frac{\tilde p(2)\,q(1\mid2)}{\tilde p(1)\,q(2\mid1)}=\frac{e^{-2}\cdot\frac12e^{-0.5}}{e^{-1}\cdot e^{-2}}=\tfrac12e^{0.5}\approx0.824$, so $\alpha\approx0.824$.
Without the correction $\alpha=e^{-1}\approx0.368$. Because this proposal tends to propose larger values from larger states, ignoring $q$ would sample the wrong distribution.

```python
acc_mh = 0.5 * np.exp(0.5)
acc_wrong = np.exp(-1)
```

</details>

### Exercise 7 · Detailed balance
*✍️ By hand · ★★☆*

Show that the MH transition kernel $T(\theta\to\theta')=q(\theta'\mid\theta)\,\alpha(\theta,\theta')$ (for $\theta'\ne\theta$) satisfies **detailed balance**
$p(\theta)T(\theta\to\theta')=p(\theta')T(\theta'\to\theta)$, and that detailed balance implies $p$ is a stationary distribution.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Use $\min(a,b)$ symmetry: $p(\theta)q(\theta'\mid\theta)\min\!\big(1,\frac{p(\theta')q(\theta\mid\theta')}{p(\theta)q(\theta'\mid\theta)}\big)=\min\big(p(\theta)q(\theta'\mid\theta),\ p(\theta')q(\theta\mid\theta')\big)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$p(\theta)q(\theta'\mid\theta)\alpha(\theta,\theta')=\min\{p(\theta)q(\theta'\mid\theta),\,p(\theta')q(\theta\mid\theta')\}$, which is symmetric in $(\theta,\theta')$, hence equals $p(\theta')q(\theta\mid\theta')\alpha(\theta',\theta)$. (The rejection mass stays at $\theta$ and trivially balances.)
Integrating detailed balance over $\theta$: $\int p(\theta)T(\theta\to\theta')d\theta=p(\theta')\int T(\theta'\to\theta)d\theta=p(\theta')$, so one step of the chain leaves $p$ unchanged: $p$ is stationary. Together with irreducibility and aperiodicity the chain converges to $p$.

</details>

### Exercise 8 · Gibbs conditionals for a bivariate Gaussian
*✍️ By hand · ★★☆*

For $(x_1,x_2)\sim\mathcal N\!\left(0,\begin{pmatrix}1&\rho\\\rho&1\end{pmatrix}\right)$, derive $p(x_1\mid x_2)$. With $\rho=0.8$ and $x_2=1$ give its mean and variance.
Why is Gibbs an MH method with acceptance probability 1?

In [ ]:
cond_mean = None
cond_var = None

_S = np.array([[1, .8], [.8, 1]])
check('conditional mean', cond_mean, _S[0, 1] / _S[1, 1] * 1.0)
check('conditional variance', cond_var, _S[0, 0] - _S[0, 1] ** 2 / _S[1, 1])

<details>
<summary><b>💡 Hint</b></summary>

Gaussian conditioning: $\mu_{1\mid2}=\mu_1+\Sigma_{12}\Sigma_{22}^{-1}(x_2-\mu_2)$, $\Sigma_{1\mid2}=\Sigma_{11}-\Sigma_{12}\Sigma_{22}^{-1}\Sigma_{21}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$x_1\mid x_2\sim\mathcal N(\rho x_2,\,1-\rho^2)$: mean $0.8$, variance $0.36$.
Gibbs proposes $x_1'\sim p(x_1\mid x_2)$ and keeps $x_2$. Plugging $q=p(x_1'\mid x_2)$ into the MH ratio gives $\frac{p(x_1',x_2)\,p(x_1\mid x_2)}{p(x_1,x_2)\,p(x_1'\mid x_2)}=\frac{p(x_2)}{p(x_2)}=1$. As $\rho\to1$ the conditionals become very narrow and Gibbs crawls along the diagonal (Exercise 13).

```python
cond_mean = 0.8
cond_var = 0.36
```

</details>

### Exercise 9 · Effective sample size of an AR(1) chain
*✍️ By hand · ★★☆*

A stationary chain has autocorrelation $\rho_k=\rho^k$. The ESS is $N/\big(1+2\sum_{k\ge1}\rho_k\big)$. Derive the closed form and evaluate it
for $N=10\,000$ and $\rho=0.9$. What is the Monte Carlo standard error of the posterior mean if the posterior std is 1?

In [ ]:
ess_ar1 = None
mcse = None

_k = np.arange(1, 2000)
check('ESS', ess_ar1, 10000 / (1 + 2 * np.sum(0.9 ** _k)), tol=1e-6)
check('MCSE', mcse, 1 / np.sqrt(10000 / (1 + 2 * np.sum(0.9 ** _k))), tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

Geometric series: $\sum_{k\ge1}\rho^k=\frac{\rho}{1-\rho}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$1+\frac{2\rho}{1-\rho}=\frac{1+\rho}{1-\rho}$, so ESS $=N\frac{1-\rho}{1+\rho}=10000\cdot\frac{0.1}{1.9}\approx526$.
MCSE $=\sigma/\sqrt{\text{ESS}}\approx1/\sqrt{526}\approx0.044$, instead of $0.01$ for 10 000 independent draws.

```python
ess_ar1 = 10000 * 0.1 / 1.9
mcse = 1 / np.sqrt(ess_ar1)
```

</details>

### Exercise 10 · R-hat by hand
*✍️ By hand · ★★☆*

Two chains of $n=100$ draws each have means $0.0$ and $1.0$ and within-chain variances $1.0$ and $1.0$. Compute
$W$ (mean within-chain variance), $B=n\cdot\mathrm{Var}(\text{chain means})$ (with $m-1$ in the denominator),
$\widehat{\mathrm{var}}^+=\frac{n-1}nW+\frac1nB$ and $\hat R=\sqrt{\widehat{\mathrm{var}}^+/W}$. Is it acceptable by the note's threshold?

In [ ]:
R_hat_hand = None

_m = np.array([0., 1.]); _W = 1.0; _B = 100 * _m.var(ddof=1)
check('R-hat', R_hat_hand, np.sqrt((99 / 100 * _W + _B / 100) / _W))

<details>
<summary><b>💡 Hint</b></summary>

$\mathrm{Var}(0,1)$ with $m-1=1$ in the denominator is $0.5$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$W=1$, $B=100\cdot0.5=50$, $\widehat{\mathrm{var}}^+=0.99+0.5=1.49$, $\hat R=\sqrt{1.49}\approx1.22$: far above $1.01$, the chains disagree and have not converged to the same distribution.

```python
R_hat_hand = np.sqrt(1.49)
```

</details>

## Part C · Samplers from scratch

### Exercise 11 · Random-walk Metropolis
*💻 Code · ★☆☆*

Implement `rw_metropolis(log_p, theta0, n, step, rng)` returning `(samples, acceptance_rate)`, working with **log** densities.
Sample the shared target `log_p_tilde` (a $\mathrm{Beta}(9,5)$ in disguise) with step 0.2, 20 000 iterations, discard 1 000 as burn-in,
and compare the sample mean and std with the exact values. Plot the trace and a histogram against `stats.beta(9, 5).pdf`.

In [ ]:
def rw_metropolis(log_p, theta0, n, step, rng):
    # TODO: accept if log(u) < log_p(prop) - log_p(current)
    return None, None

samples, acc = rw_metropolis(log_p_tilde, 0.5, 20000, 0.2, np.random.default_rng(1))
post = None if samples is None else samples[1000:]

check('posterior mean', None if post is None else post.mean(), stats.beta(9, 5).mean(), tol=0.01)
check('posterior std', None if post is None else post.std(), stats.beta(9, 5).std(), tol=0.01)

<details>
<summary><b>💡 Hint</b></summary>

Proposals outside $(0,1)$ get $\log\tilde p=-\infty$ and are automatically rejected. On rejection, **repeat** the current value in the output.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Mean $\approx0.643$, std $\approx0.124$, matching $\mathrm{Beta}(9,5)$, although we never used its normalizing constant. Forgetting to record the current state after a rejection is the most common bug: it biases the sampler.

```python
def rw_metropolis(log_p, theta0, n, step, rng):
    theta, lp = theta0, log_p(theta0)
    out = np.empty(n); acc = 0
    for i in range(n):
        prop = theta + step * rng.normal()
        lp_prop = log_p(prop)
        if np.log(rng.random()) < lp_prop - lp:
            theta, lp = prop, lp_prop
            acc += 1
        out[i] = theta
    return out, acc / n

samples, acc = rw_metropolis(log_p_tilde, 0.5, 20000, 0.2, np.random.default_rng(1))
post = samples[1000:]
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(samples[:2000], lw=0.5); ax[0].set_title(f'trace (acc={acc:.2f})')
ax[1].hist(post, 60, density=True, alpha=.5); _g = np.linspace(0, 1, 200); ax[1].plot(_g, stats.beta(9, 5).pdf(_g))
plt.show()
```

</details>

### Exercise 12 · Acceptance rate vs step size
*💻 Code · ★★☆*

Run your sampler on a standard normal target (`lambda t: -0.5 * t**2`) with steps $s\in\{0.05, 0.5, 2.4, 10, 50\}$ (10 000 iterations each).
Store the acceptance rates in `acc_rates` and the lag-1 autocorrelations in `lag1`. Which step gives the lowest lag-1 autocorrelation?

In [ ]:
steps = [0.05, 0.5, 2.4, 10, 50]
acc_rates = None
lag1 = None

check('acceptance decreases with step', None if acc_rates is None else bool(np.all(np.diff(acc_rates) < 0)), True)
check('a medium step mixes best', None if lag1 is None else int(np.argmin(lag1)) in (2, 3), True)

<details>
<summary><b>💡 Hint</b></summary>

Lag-1 autocorrelation: `np.corrcoef(x[:-1], x[1:])[0, 1]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Acceptance falls from $\approx0.98$ to $\approx0.02$. Lag-1 autocorrelation is near 1 for both $s=0.05$ (tiny moves) and $s=50$ (stuck), and lowest around $s\approx2.4$ (acceptance $\approx0.45$ in 1-D), confirming the trade-off from Exercise 3.

```python
steps = [0.05, 0.5, 2.4, 10, 50]
acc_rates, lag1 = [], []
for s in steps:
    x, a = rw_metropolis(lambda t: -0.5 * t ** 2, 0.0, 10000, s, np.random.default_rng(2))
    acc_rates.append(a); lag1.append(np.corrcoef(x[:-1], x[1:])[0, 1])
print(np.round(acc_rates, 3)); print(np.round(lag1, 3))
```

</details>

### Exercise 13 · A Gibbs sampler
*💻 Code · ★★☆*

Implement a Gibbs sampler for the bivariate Gaussian of Exercise 8 with $\rho=0.8$ (alternately sample $x_1\mid x_2$ and $x_2\mid x_1$),
10 000 sweeps from $(5,-5)$. Store the samples after 500 burn-in sweeps in `gibbs` (shape $(n,2)$). Then try $\rho=0.995$ and look at the trace.

In [ ]:
gibbs = None

check('sample correlation', None if gibbs is None else np.corrcoef(gibbs.T)[0, 1], 0.8, tol=0.03)
check('sample variances', None if gibbs is None else gibbs.var(0), [1, 1], tol=0.08)

<details>
<summary><b>💡 Hint</b></summary>

$x_1\leftarrow\rho x_2+\sqrt{1-\rho^2}\,\varepsilon$, then $x_2\leftarrow\rho x_1+\sqrt{1-\rho^2}\,\varepsilon'$ using the *new* $x_1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With $\rho=0.8$ the sample correlation is $\approx0.8$. With $\rho=0.995$ each conditional has std $\approx0.1$, so the chain moves along the diagonal in tiny axis-aligned steps and mixes very slowly. Reparameterizing (e.g. rotating to $x_1\pm x_2$) or blocking both variables fixes it.

```python
def gibbs_bvn(rho, n, x0, rng):
    x1, x2 = x0
    out = np.empty((n, 2)); sd = np.sqrt(1 - rho ** 2)
    for i in range(n):
        x1 = rho * x2 + sd * rng.normal()
        x2 = rho * x1 + sd * rng.normal()
        out[i] = x1, x2
    return out

gibbs = gibbs_bvn(0.8, 10000, (5.0, -5.0), np.random.default_rng(3))[500:]
slow = gibbs_bvn(0.995, 3000, (5.0, -5.0), np.random.default_rng(3))
plt.plot(slow[:, 0], lw=.5); plt.title('rho=0.995: slow mixing'); plt.show()
```

</details>

### Exercise 14 · Estimating ESS from autocorrelations
*💻 Code · ★★☆*

Implement `ess(x)`: estimate autocorrelations $\hat\rho_k$ (FFT or direct), sum them in **pairs** $\hat\rho_{2t}+\hat\rho_{2t+1}$ and stop at the first
negative pair (Geyer's initial positive sequence). Test on a simulated AR(1) chain with $\rho=0.9$, $N=10\,000$ and compare with Exercise 9.

In [ ]:
def ess(x):
    # TODO
    return None

r7 = np.random.default_rng(4)
ar = np.empty(10000); ar[0] = r7.normal() / np.sqrt(1 - 0.81)
for i in range(1, 10000):
    ar[i] = 0.9 * ar[i - 1] + r7.normal()
ess_ar = ess(ar)

check('ESS close to N(1-rho)/(1+rho)', None if ess_ar is None else abs(ess_ar / (10000 * 0.1 / 1.9) - 1) < 0.25, True)
check('ESS of iid draws is about N', None if ess(np.random.default_rng(5).normal(size=5000)) is None else abs(ess(np.random.default_rng(5).normal(size=5000)) / 5000 - 1) < 0.15, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Direct autocovariance: `c_k = np.mean((x[:-k] - m) * (x[k:] - m))` for $k\ge1$, divided by the variance.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

ESS $=N/(-1+2\sum_t P_t)$ with $P_t=\hat\rho_{2t}+\hat\rho_{2t+1}$, $\hat\rho_0=1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Since $\hat\rho_0=1$: $1+2\sum_{k\ge1}\hat\rho_k=-1+2\sum_{t\ge0}P_t$. Truncating at the first negative pair stops summing pure noise at large lags, which would otherwise make the estimate erratic.
For this chain the estimate is $\approx450$, within the sampling noise of the theoretical 526 (ESS estimates of strongly autocorrelated chains are themselves noisy). Stan and ArviZ use this estimator (plus multi-chain refinements).

```python
def ess(x):
    x = np.asarray(x, float); n = len(x); m = x.mean(); v = x.var()
    f = np.fft.rfft(x - m, 2 * n)
    acov = np.fft.irfft(f * np.conj(f))[:n] / n
    rho = acov / v
    s, t = 0.0, 0
    while 2 * t + 1 < n:
        P = rho[2 * t] + rho[2 * t + 1]
        if P < 0:
            break
        s += P; t += 1
    return n / (-1 + 2 * s)

r7 = np.random.default_rng(4)
ar = np.empty(10000); ar[0] = r7.normal() / np.sqrt(1 - 0.81)
for i in range(1, 10000):
    ar[i] = 0.9 * ar[i - 1] + r7.normal()
ess_ar = ess(ar)
print(ess_ar)
```

</details>

### Exercise 15 · Computing R-hat for several chains
*💻 Code · ★★☆*

Implement `rhat(chains)` for an array of shape `(m, n)` using the formulas of Exercise 10. Evaluate it on (a) 4 chains of your Metropolis
sampler on `log_p_tilde` started at $0.05,0.35,0.65,0.95$ (2 000 draws each, after 500 burn-in) and (b) 4 "stuck" chains, simulated as
$\mathcal N(\mu_j,1)$ draws with $\mu_j=0,0,0,2$. Store `rhat_good` and `rhat_bad`.

In [ ]:
def rhat(chains):
    # TODO
    return None

rhat_good = None
rhat_bad = None

check('R-hat function on the two-chain example', None if rhat(np.array([[-1., 1.], [0., 2.]])) is None else rhat(np.array([[-1., 1.], [0., 2.]])), np.sqrt((0.5 * 2 + 0.5) / 2))
check('good chains < 1.01, bad chains > 1.1', None if rhat_good is None else (rhat_good < 1.01 and rhat_bad > 1.1), True)

<details>
<summary><b>💡 Hint</b></summary>

`W = chains.var(axis=1, ddof=1).mean()`, `B = n * chains.mean(axis=1).var(ddof=1)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Good chains: $\hat R\approx1.00$. Stuck chains: $\hat R\approx1.4$, because one chain's mean differs by about 2 within-chain standard deviations.
Modern practice (Vehtari et al. 2021) uses rank-normalized, split-chain $\hat R$, which also catches chains that drift within themselves.

```python
def rhat(chains):
    chains = np.asarray(chains, float); m, n = chains.shape
    W = chains.var(axis=1, ddof=1).mean()
    B = n * chains.mean(axis=1).var(ddof=1)
    return np.sqrt(((n - 1) / n * W + B / n) / W)

good = np.array([rw_metropolis(log_p_tilde, t0, 2500, 0.2, np.random.default_rng(10 + j))[0][500:]
                 for j, t0 in enumerate([0.05, 0.35, 0.65, 0.95])])
rhat_good = rhat(good)
_r = np.random.default_rng(11)
rhat_bad = rhat(np.array([_r.normal(mu, 1, 2000) for mu in (0, 0, 0, 2)]))
print(rhat_good, rhat_bad)
```

</details>

### Exercise 16 · Rejection and importance sampling
*💻 Code · ★★☆*

Target $\mathrm{Beta}(9,5)$ via `log_p_tilde`, proposal $q=\mathrm{Uniform}(0,1)$, 100 000 proposals.
(a) **Rejection**: with $M=\max_\theta\tilde p(\theta)$, accept $\theta$ if $u<\tilde p(\theta)/M$. Store the acceptance rate `acc_rej` and the mean of accepted draws `mean_rej`.
(b) **Self-normalized importance sampling**: weights $w\propto\tilde p(\theta)/q(\theta)$; store the estimate `mean_is` and the effective sample size $(\sum w)^2/\sum w^2$ in `ess_is`.

In [ ]:
acc_rej = None
mean_rej = None
mean_is = None
ess_is = None

_b = stats.beta(9, 5)
check('rejection acceptance = 1 / sup(p/q) = 1 / pdf(mode)', acc_rej, 1 / _b.pdf(8 / 12), tol=5e-3)
check('rejection mean', mean_rej, _b.mean(), tol=5e-3)
check('IS mean', mean_is, _b.mean(), tol=5e-3)

<details>
<summary><b>💡 Hint</b></summary>

The maximum of $\tilde p$ is at the mode $\theta=8/12$. The acceptance rate is $Z/M$ for a uniform proposal.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Acceptance $\approx1/p(\text{mode})\approx0.33$ (2/3 of proposals wasted); both estimates are $\approx0.643$. Importance sampling keeps every draw, with an ESS of $\approx44\%$ of the draws.
In $d$ dimensions with a product target, the acceptance rate becomes $\approx0.33^d$: $10^{-5}$ for $d=10$. This is why the note calls them poor in high dimension.

```python
_r = np.random.default_rng(6)
th = _r.random(100000)
lp = log_p_tilde(th)
lM = log_p_tilde(8 / 12)
accept = np.log(_r.random(100000)) < lp - lM
acc_rej, mean_rej = accept.mean(), th[accept].mean()
w = np.exp(lp - lp.max())
mean_is = np.sum(w * th) / w.sum()
ess_is = w.sum() ** 2 / np.sum(w ** 2)
print(acc_rej, mean_rej, mean_is, ess_is)
```

</details>

### Exercise 17 · Hamiltonian Monte Carlo from scratch
*💻 Code · ★★★*

Implement HMC for the strongly correlated 2-D Gaussian $\mathcal N(0,\Sigma)$, $\Sigma=\begin{pmatrix}1&0.95\\0.95&1\end{pmatrix}$:

1. draw momentum $r\sim\mathcal N(0,I)$;
2. run $L=20$ **leapfrog** steps of size $\epsilon=0.1$ on $H(\theta,r)=-\log p(\theta)+\tfrac12r^\top r$;
3. accept with probability $\min(1,e^{H_{old}-H_{new}})$.

Draw 3 000 samples (after 200 warm-up) and store them in `hmc_samples` and the acceptance rate in `hmc_acc`. Compare the ESS of $\theta_1$
with random-walk Metropolis using the same number of **gradient/density evaluations**.

In [ ]:
Sigma_h = np.array([[1, 0.95], [0.95, 1]]); Prec_h = np.linalg.inv(Sigma_h)
log_p_h = lambda th: -0.5 * th @ Prec_h @ th
grad_log_p_h = lambda th: -Prec_h @ th
def hmc(n, eps=0.1, L=20, seed=0):
    # TODO: return samples (n, 2), acceptance rate
    return None, None

hmc_samples, hmc_acc = hmc(3200)
hmc_samples = None if hmc_samples is None else hmc_samples[200:]

check('sample covariance', None if hmc_samples is None else np.cov(hmc_samples.T), Sigma_h, tol=0.15)
check('high acceptance', None if hmc_acc is None else hmc_acc > 0.8, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Leapfrog: $r\leftarrow r+\tfrac\epsilon2\nabla\log p(\theta)$; then $L$ times $\theta\leftarrow\theta+\epsilon r$ and a full momentum step (half step at the very end).

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Leapfrog is reversible and volume preserving, so no Hastings correction for $q$ is needed; the energy error is $O(\epsilon^2)$, hence high acceptance.

</details>

<details>
<summary><b>✅ Solution</b></summary>

HMC accepts $>90\%$ of its long trajectories and travels along the narrow ridge in a few iterations, so the ESS is high (here $\approx2200$ of 3 000 draws).
Random-walk Metropolis with the same budget of 64 000 density evaluations must use small steps (the ridge is only $\sqrt{1-0.95^2}\approx0.3$ wide) and ends up with a much smaller ESS (here $\approx300$).
Too large $\epsilon$ makes the leapfrog integrator unstable (acceptance collapses); NUTS adapts $L$ and $\epsilon$ automatically.

```python
Sigma_h = np.array([[1, 0.95], [0.95, 1]]); Prec_h = np.linalg.inv(Sigma_h)
log_p_h = lambda th: -0.5 * th @ Prec_h @ th
grad_log_p_h = lambda th: -Prec_h @ th
def hmc(n, eps=0.1, L=20, seed=0):
    r_ = np.random.default_rng(seed)
    th = np.array([2.0, -2.0]); out = np.empty((n, 2)); acc = 0
    for i in range(n):
        r0 = r_.normal(size=2)
        t, r = th.copy(), r0 + 0.5 * eps * grad_log_p_h(th)
        for l in range(L):
            t = t + eps * r
            if l < L - 1:
                r = r + eps * grad_log_p_h(t)
        r = r + 0.5 * eps * grad_log_p_h(t)
        H_old = -log_p_h(th) + 0.5 * r0 @ r0
        H_new = -log_p_h(t) + 0.5 * r @ r
        if np.log(r_.random()) < H_old - H_new:
            th = t; acc += 1
        out[i] = th
    return out, acc / n

hmc_samples, hmc_acc = hmc(3200)
hmc_samples = hmc_samples[200:]
# Random-walk Metropolis with the same number of evaluations (20 per HMC iteration)
_r = np.random.default_rng(1); th = np.zeros(2); rw = np.empty((64000, 2)); lp = log_p_h(th)
for i in range(64000):
    prop = th + 0.25 * _r.normal(size=2); lpp = log_p_h(prop)
    if np.log(_r.random()) < lpp - lp:
        th, lp = prop, lpp
    rw[i] = th
print('HMC acc', hmc_acc, '| ESS HMC', round(ess(hmc_samples[:, 0])), '| ESS RWM (64k steps)', round(ess(rw[:, 0])))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [MCMC](MCMC.md).*